# 🎙️ 声音克隆：Qwen3-TTS 1.7B（Colab 免费 T4）

配合「声音克隆工作台」使用：先在工作台录好声音、输好文字，下载 **voice-pack.zip**，再按下面顺序运行。

**开始前：运行时 → 更改运行时类型 → 硬件加速器选 T4 GPU。**

流程：① 装依赖 → ② 上传语音包 → ③ 加载模型（首次约 1–2 分钟，3.86GB）→ ④ 逐段生成 → ⑤ 打包下载。

In [ ]:
# ① 检查 GPU + 安装依赖
import torch
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('显卡:', torch.cuda.get_device_name(0))
else:
    print('⚠️ 没连上 GPU！先点 运行时 → 更改运行时类型 → 选 T4 GPU，再重跑。')

# 安装 Qwen3-TTS 官方包（Colab 自带 torch，不用重装）
# 注意：diffusers 0.42.1 要求 huggingface-hub>=1.32.0，Colab 预装的 0.36.2 会冲突，必须一起升级
!pip install -U -q qwen-tts soundfile huggingface-hub

# 验证关键依赖版本
import huggingface_hub, diffusers
print('huggingface-hub:', huggingface_hub.__version__)
print('diffusers:', diffusers.__version__)

In [ ]:
# ② 上传语音包（voice-pack.zip，在弹出的窗口里选文件）
import os, zipfile, json
from google.colab import files

uploaded = files.upload()  # 选 voice-pack.zip
zip_name = next(iter(uploaded))
os.makedirs('voice_pack', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('voice_pack')
print('解压完成：', sorted(os.listdir('voice_pack')))
pack = json.load(open('voice_pack/segments.json', encoding='utf-8'))
ref_text = open('voice_pack/ref_text.txt', encoding='utf-8').read().strip()
print('声音：', pack.get('voice_name'), '| 段数：', len(pack['segments']), '| 原文已填：', bool(ref_text))

In [ ]:
# ③ 加载模型（Qwen3-TTS-12Hz-1.7B-Base，克隆专用）
# T4 是老架构：用 float16 + sdpa（不要 flash-attn，T4 上容易装不上）
import torch
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    'Qwen/Qwen3-TTS-12Hz-1.7B-Base',
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print('模型加载完成')

In [ ]:
# ④ 逐段生成（每段约几十秒）
import time, soundfile as sf

os.makedirs('output', exist_ok=True)
for seg in pack['segments']:
    i, text = seg['index'], seg['text']
    t0 = time.time()
    kwargs = dict(text=text, language='Chinese', ref_audio='voice_pack/ref.wav')
    if ref_text:
        kwargs['ref_text'] = ref_text
    else:
        # 没填原文：只用说话人向量克隆（效果稍差，但不用原文）
        kwargs['x_vector_only_mode'] = True
        kwargs['ref_text'] = None
    wavs, sr = model.generate_voice_clone(**kwargs)
    out = f'output/voice_{i:02d}.wav'
    sf.write(out, wavs[0], sr)
    print(f'第 {i}/{len(pack["segments"])} 段完成：{out}（{time.time()-t0:.0f} 秒）')
print('全部生成完成')

In [ ]:
# ⑤ 打包下载
import shutil
shutil.make_archive('voice-output', 'zip', 'output')
files.download('voice-output.zip')
print('已下载 voice-output.zip，每段一个 wav')